## Imports

In [1]:
import os
import re
import time
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 50)

In [3]:
## Connection
connection = presto.connect(
        
        host='presto-gateway.processing.data.production.internal',
#     presto.processing.yoda.run
#     presto-gateway.processing.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
cwd = os.getcwd()
print(cwd)
local_datasource = '/Users/E2074/local-datasets/customer/ads-monetisation/holdout-group'
print(local_datasource)

/Users/E2074/analytics/customer/Ads-Monetisation/holdout-group/jas
/Users/E2074/local-datasets/customer/ads-monetisation/holdout-group


## Datasets & Parameter

In [5]:
## Parameter 
current_date = datetime.now()
start_date = '20240415'
end_date = '20240715'

# Convert date strings to datetime objects
start_dt = datetime.strptime(start_date, '%Y%m%d')
end_dt = datetime.strptime(end_date, '%Y%m%d')
segment_date = end_dt.strftime('%Y-%m-%d')
print(start_date, ' to ' ,end_date)

20240415  to  20240715


### AMJ Gross Customers

In [7]:
data_query = f"""

    WITH customer_base as (
    
    SELECT 
        customer_id,
        city_name,
        customer_obj_mobile customer_mobile,
        COUNT(DISTINCT order_id) net_orders,
        COUNT(DISTINCT CASE WHEN service_obj_service_name IN ('Link','Bike Lite','Bike Pink','Bike Metro') THEN order_id END) bike_service,
        COUNT(DISTINCT CASE WHEN service_obj_service_name IN ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto') THEN order_id END) auto_service,
        COUNT(DISTINCT CASE WHEN service_obj_service_name IN ('CabEconomy', 'CabPremium') THEN order_id END) cab_service
    FROM    
        orders.order_logs_snapshot
    WHERE  
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND order_status = 'dropped'
        AND city_name IN ('Chennai', 'Delhi', 'Jaipur', 'Ahmedabad')
        AND service_obj_service_name in ('Auto','Auto Lite','Auto NCR','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','Bike Metro','CabEconomy','CabPremium')
    GROUP BY 1,2,3
    ),
    
    segment AS (
    
    SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_retention_segments retention_segment,
        taxi_lifetime_stage lifetime_stage,
        taxi_service_affinity service_affinity,
        gender_tag gender
    FROM
        datasets.iallocator_customer_segments
    WHERE 
        run_date = '{segment_date}'
    )
    
    SELECT 
        customer_base.*,
        coalesce( ltr_segment, 'UNKNOWN')ltr_segment,
        coalesce( retention_segment, 'UNKNOWN') retention_segment,
        coalesce( lifetime_stage, 'UNKNOWN') lifetime_stage,
        coalesce( service_affinity, 'UNKNOWN') service_affinity,
        coalesce( gender, 'UNKNOWN') gender
    FROM 
        customer_base
    LEFT JOIN 
        segment
        ON customer_base.customer_id = segment.customer_id
"""

# Execute the query and get the result as a DataFrame
df_data = pd.read_sql(data_query, connection)

df_data.to_csv( local_datasource + '/customer_data_{}_{}.csv'.format(start_date, end_date), index=False)

In [ ]:
# df_data = pd.read_csv( local_datasource + '/analysis_data_{}_{}.csv'.format(start_date, end_date))

In [8]:
df_data.head(5)

,customer_id,city_name,customer_mobile,net_orders,bike_service,auto_service,cab_service,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender
0,64c880880b7df873f86b8496,Chennai,9790734128,26,0,26,0,PHH,ELITE,COMMITTED,ONLY_AUTO,FEMALE
1,64988c1204fb2b38387a2c30,Jaipur,9749865747,16,13,3,0,PHH,PLATINUM,COMMITTED,ONLY_LINK,MALE
2,63b575fc6b2244b1bb939196,Delhi,9310417485,3,3,0,0,PHH,SILVER,SUSTENANCE,UNKNOWN,FEMALE
3,610df2f1e21262f12cfd4f31,Delhi,9534528247,9,9,0,0,PHH,GOLD,HOOK,ONLY_LINK,MALE
4,5c6e934735478b560278068b,Delhi,7980759744,61,21,32,8,PHH,ELITE,COMMITTED,ONLY_CAB,MALE


## User defined function

In [9]:
agg_dict = {
    'customers': ('customer_id', 'nunique'),
    'net_orders': ('net_orders', 'sum'),
    'bike_service': ('bike_service', 'sum'),
    'auto_service': ('auto_service', 'sum'),
    'cab_service': ('cab_service', 'sum'),
}

def calculate_percentage(df, numerator, denominator):
    
    percentage = (df[numerator] * 100.00 / df[denominator]).round(2)
    
    return percentage

def calculate_percentage_columns(df):
        
    df['bike'] = calculate_percentage(df, 'bike_service', 'net_orders')
    df['auto'] = calculate_percentage(df, 'auto_service', 'net_orders')
    df['cab'] = calculate_percentage(df, 'cab_service', 'net_orders')
    
    return df

## Sampling

In [10]:
df_data.head(2)

,customer_id,city_name,customer_mobile,net_orders,bike_service,auto_service,cab_service,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender
0,64c880880b7df873f86b8496,Chennai,9790734128,26,0,26,0,PHH,ELITE,COMMITTED,ONLY_AUTO,FEMALE
1,64988c1204fb2b38387a2c30,Jaipur,9749865747,16,13,3,0,PHH,PLATINUM,COMMITTED,ONLY_LINK,MALE


In [11]:
df_data.columns

Index(['customer_id', 'city_name', 'customer_mobile', 'net_orders',
       'bike_service', 'auto_service', 'cab_service', 'ltr_segment',
       'retention_segment', 'lifetime_stage', 'service_affinity', 'gender'],
      dtype='object')

In [18]:
df_analysis2 = df_data\
.groupby(['city_name'])\
.agg(**agg_dict)\
.reset_index()

df_analysis2 = calculate_percentage_columns(df_analysis2)
df_analysis2

,city_name,customers,net_orders,bike_service,auto_service,cab_service,bike,auto,cab
0,Ahmedabad,643527,3512985,2014874,1054814,443297,57.36,30.03,12.62
1,Chennai,2587273,18905963,7390618,10283251,1232094,39.09,54.39,6.52
2,Delhi,3721059,24557802,15622879,5596005,3338918,63.62,22.79,13.60
3,Jaipur,889675,3995662,2446977,1528861,19824,61.24,38.26,0.50


In [19]:
df_analysis_6 = df_data\
.groupby(['ltr_segment'])\
.agg(**agg_dict)\
.reset_index()

df_analysis_6 = calculate_percentage_columns(df_analysis_6)
df_analysis_6

,ltr_segment,customers,net_orders,bike_service,auto_service,cab_service,bike,auto,cab
0,HH,2478626,3947250,2120378,1221153,605719,53.72,30.94,15.35
1,PHH,5251615,47013671,25346044,17239293,4428334,53.91,36.67,9.42
2,UNKNOWN,10744,11491,8926,2485,80,77.68,21.63,0.70


In [20]:
df_analysis_7 = df_data\
.groupby(['retention_segment'])\
.agg(**agg_dict)\
.reset_index()

df_analysis_7 = calculate_percentage_columns(df_analysis_7)
df_analysis_7

,retention_segment,customers,net_orders,bike_service,auto_service,cab_service,bike,auto,cab
0,DORMANT,3757184,11327478,6112210,3929673,1285595,53.96,34.69,11.35
1,ELITE,399899,17944140,10006850,6541594,1395696,55.77,36.46,7.78
2,GOLD,1009367,6451491,3257985,2443632,749874,50.50,37.88,11.62
3,HH,848954,1519604,823540,466532,229532,54.19,30.70,15.10
4,INACTIVE,77249,91670,55898,26409,9363,60.98,28.81,10.21
5,PLATINUM,731701,10529361,5208237,4212089,1109035,49.46,40.00,10.53
6,PRIME,22915,943517,827927,90318,25272,87.75,9.57,2.68
7,SILVER,882972,2153660,1173775,750199,229686,54.50,34.83,10.66
8,UNKNOWN,10744,11491,8926,2485,80,77.68,21.63,0.70


In [21]:
df_analysis_8 = df_data\
.groupby(['lifetime_stage'])\
.agg(**agg_dict)\
.reset_index()

df_analysis_8 = calculate_percentage_columns(df_analysis_8)
df_analysis_8

,lifetime_stage,customers,net_orders,bike_service,auto_service,cab_service,bike,auto,cab
0,CHURN_OTB,255190,3026266,1633186,1107557,285523,53.97,36.60,9.43
1,COMMITTED,812140,23567699,12887953,8664726,2015020,54.68,36.77,8.55
2,DETOX,15249,177632,154916,17602,5114,87.21,9.91,2.88
3,DORMANT,3757184,11327478,6112210,3929673,1285595,53.96,34.69,11.35
4,HANDHOLDING,848954,1519604,823540,466532,229532,54.19,30.70,15.10
5,HOOK,1286083,6881249,3559639,2602987,718623,51.73,37.83,10.44
6,INACTIVE,77249,91670,55898,26409,9363,60.98,28.81,10.21
7,SOFT_CHURN,191793,1248380,669881,453508,124991,53.66,36.33,10.01
8,SUSTENANCE,486399,3120943,1569199,1191452,360292,50.28,38.18,11.54
9,UNKNOWN,10744,11491,8926,2485,80,77.68,21.63,0.70


In [22]:
# df_analysis_9 = df_data\
# .groupby(['service_affinity'])\
# .agg(**agg_dict)\
# .reset_index()

# df_analysis_9 = calculate_percentage_columns(df_analysis_9)
# df_analysis_9

In [23]:
df_analysis_10 = df_data\
.groupby(['gender'])\
.agg(**agg_dict)\
.reset_index()

df_analysis_10 = calculate_percentage_columns(df_analysis_10)
df_analysis_10

,gender,customers,net_orders,bike_service,auto_service,cab_service,bike,auto,cab
0,FEMALE,1767037,15709943,5698937,8352413,1658593,36.28,53.17,10.56
1,MALE,5785009,34039072,21148064,9615641,3275367,62.13,28.25,9.62
2,OTHERS,19922,138569,68965,56123,13481,49.77,40.50,9.73
3,UNKNOWN,169017,1084828,559382,438754,86692,51.56,40.44,7.99


In [25]:
df_analysis_11 = df_data\
.groupby(['city_name','ltr_segment'])\
.agg(**agg_dict)\
.reset_index()

df_analysis_11 = calculate_percentage_columns(df_analysis_11)
df_analysis_11

,city_name,ltr_segment,customers,net_orders,bike_service,auto_service,cab_service,bike,auto,cab
0,Ahmedabad,HH,243666,389820,212407,103912,73501,54.49,26.66,18.86
1,Ahmedabad,PHH,398243,3121503,1801061,950646,369796,57.70,30.45,11.85
2,Ahmedabad,UNKNOWN,1618,1662,1406,256,0,84.60,15.40,0.00
3,Chennai,HH,656815,1074359,417664,551292,105403,38.88,51.31,9.81
4,Chennai,PHH,1928375,17829483,6971484,9731309,1126690,39.10,54.58,6.32
5,Chennai,UNKNOWN,2083,2121,1470,650,1,69.31,30.65,0.05
6,Delhi,HH,1304334,2050082,1224779,402216,423087,59.74,19.62,20.64
7,Delhi,PHH,2411053,22501638,14393052,5192834,2915752,63.96,23.08,12.96
8,Delhi,UNKNOWN,5672,6082,5048,955,79,83.00,15.70,1.30
9,Jaipur,HH,283093,432989,265528,163733,3728,61.32,37.81,0.86


In [28]:
df_analysis_15 = df_data\
.groupby(['city_name', 'retention_segment'])\
.agg(**agg_dict)\
.reset_index()

df_analysis_15 = calculate_percentage_columns(df_analysis_15)
df_analysis_15

,city_name,retention_segment,customers,net_orders,bike_service,auto_service,cab_service,bike,auto,cab
0,Ahmedabad,DORMANT,324645,921098,512280,274871,133947,55.62,29.84,14.54
1,Ahmedabad,ELITE,25388,1065282,647949,319209,98124,60.82,29.96,9.21
2,Ahmedabad,GOLD,79649,467963,246565,149437,71961,52.69,31.93,15.38
3,Ahmedabad,HH,84328,150701,82090,40793,27818,54.47,27.07,18.46
4,Ahmedabad,INACTIVE,7430,8877,5520,2344,1013,62.18,26.41,11.41
5,Ahmedabad,PLATINUM,50596,645494,352569,207201,85724,54.62,32.10,13.28
6,Ahmedabad,PRIME,2609,92771,81017,9300,2454,87.33,10.02,2.65
7,Ahmedabad,SILVER,67264,159137,85478,51403,22256,53.71,32.30,13.99
8,Ahmedabad,UNKNOWN,1618,1662,1406,256,0,84.60,15.40,0.00
9,Chennai,DORMANT,1188097,3882264,1523160,2046424,312680,39.23,52.71,8.05


In [29]:
df_data['city_name'].value_counts(True)

city_name
Delhi        0.474531
Chennai      0.329946
Jaipur       0.113457
Ahmedabad    0.082066
Name: proportion, dtype: float64

In [30]:
df_data.shape

(7841616, 12)

In [31]:
df_data.customer_id.nunique()

7740985

In [32]:
duplicated_values = df_data['customer_id'].duplicated(keep=False) 
df_cleaned = df_data[~duplicated_values]
df_cleaned.shape

(7642470, 12)

In [33]:
df_cleaned.customer_id.nunique()

7642470

In [34]:
random_seed = 42

df_train_control, df_test = train_test_split(df_cleaned, 
                                             test_size=0.2, 
                                             random_state=random_seed
                                            )

df_train_control = df_train_control.sample(len(df_train_control), random_state=random_seed)
df_test = df_test.sample(len(df_test), random_state=random_seed)

# Assign group labels
df_train_control['group_tc'] = 'bau'
df_test['group_tc'] = 'global-control'

df_final_sample = pd.concat([df_train_control, df_test])

In [35]:
df_final_sample['group_tc'].value_counts(True)

group_tc
bau               0.8
global-control    0.2
Name: proportion, dtype: float64

In [36]:
df_final_sample.groupby('city_name')['group_tc'].value_counts(True).unstack(fill_value=0)

group_tc,bau,global-control
city_name,,
Ahmedabad,0.800457,0.199543
Chennai,0.799869,0.200131
Delhi,0.799937,0.200063
Jaipur,0.800339,0.199661


In [37]:
df_final_sample.columns

Index(['customer_id', 'city_name', 'customer_mobile', 'net_orders',
       'bike_service', 'auto_service', 'cab_service', 'ltr_segment',
       'retention_segment', 'lifetime_stage', 'service_affinity', 'gender',
       'group_tc'],
      dtype='object')

In [38]:
df1 = df_final_sample\
.groupby(['city_name','group_tc'])\
.agg(**agg_dict)\
.reset_index()

df1 = calculate_percentage_columns(df1)
df1

,city_name,group_tc,customers,net_orders,bike_service,auto_service,cab_service,bike,auto,cab
0,Ahmedabad,bau,492539,2676265,1543377,800896,331992,57.67,29.93,12.41
1,Ahmedabad,global-control,122783,670507,384637,202529,83341,57.37,30.21,12.43
2,Chennai,bau,2049875,14997700,5849674,8176073,971953,39.00,54.52,6.48
3,Chennai,global-control,512888,3751285,1466674,2042720,241891,39.10,54.45,6.45
4,Delhi,bau,2908115,19247486,12262639,4377985,2606862,63.71,22.75,13.54
5,Delhi,global-control,727313,4813154,3067458,1093417,652279,63.73,22.72,13.55
6,Jaipur,bau,663447,2988521,1830830,1143974,13717,61.26,38.28,0.46
7,Jaipur,global-control,165510,748142,460530,284136,3476,61.56,37.98,0.46


In [39]:
df2 = df_final_sample\
.groupby(['city_name','ltr_segment', 'group_tc'])\
.agg(**agg_dict)\
.reset_index()

df2 = calculate_percentage_columns(df2)
df2

,city_name,ltr_segment,group_tc,customers,net_orders,bike_service,auto_service,cab_service,bike,auto,cab
0,Ahmedabad,HH,bau,192910,308863,168500,82242,58121,54.55,26.63,18.82
1,Ahmedabad,HH,global-control,48096,77294,41937,20820,14537,54.26,26.94,18.81
2,Ahmedabad,PHH,bau,298348,2366086,1373773,718442,273871,58.06,30.36,11.57
3,Ahmedabad,PHH,global-control,74350,592867,342398,181665,68804,57.75,30.64,11.61
4,Ahmedabad,UNKNOWN,bau,1281,1316,1104,212,0,83.89,16.11,0.00
5,Ahmedabad,UNKNOWN,global-control,337,346,302,44,0,87.28,12.72,0.00
6,Chennai,HH,bau,523639,856709,332772,440157,83780,38.84,51.38,9.78
7,Chennai,HH,global-control,131094,215024,83337,110476,21211,38.76,51.38,9.86
8,Chennai,PHH,bau,1524554,14139277,5515727,7735377,888173,39.01,54.71,6.28
9,Chennai,PHH,global-control,381393,3535854,1383042,1932133,220679,39.11,54.64,6.24


In [40]:
df3 = df_final_sample\
.groupby(['city_name','retention_segment', 'group_tc'])\
.agg(**agg_dict)\
.reset_index()

df3 = calculate_percentage_columns(df3)
df3

,city_name,retention_segment,group_tc,customers,net_orders,bike_service,auto_service,cab_service,bike,auto,cab
0,Ahmedabad,DORMANT,bau,253235,716627,399697,213596,103334,55.77,29.81,14.42
1,Ahmedabad,DORMANT,global-control,62813,177073,98506,52943,25624,55.63,29.90,14.47
2,Ahmedabad,ELITE,bau,17620,795412,489949,235996,69467,61.60,29.67,8.73
3,Ahmedabad,ELITE,global-control,4474,200414,121521,60494,18399,60.63,30.18,9.18
4,Ahmedabad,GOLD,bau,58723,358970,189388,114992,54590,52.76,32.03,15.21
...,...,...,...,...,...,...,...,...,...,...,...
67,Jaipur,PRIME,global-control,197,6075,5270,797,8,86.75,13.12,0.13
68,Jaipur,SILVER,bau,96174,223649,133406,87155,3088,59.65,38.97,1.38
69,Jaipur,SILVER,global-control,23596,54867,32473,21597,797,59.18,39.36,1.45
70,Jaipur,UNKNOWN,bau,1112,1305,823,482,0,63.07,36.93,0.00


In [41]:
df4 = df_final_sample\
.groupby(['city_name','lifetime_stage', 'group_tc'])\
.agg(**agg_dict)\
.reset_index()

df4 = calculate_percentage_columns(df4)
df4

,city_name,lifetime_stage,group_tc,customers,net_orders,bike_service,auto_service,cab_service,bike,auto,cab
0,Ahmedabad,CHURN_OTB,bau,14126,163399,94715,49960,18724,57.97,30.58,11.46
1,Ahmedabad,CHURN_OTB,global-control,3514,41073,23802,12008,5263,57.95,29.24,12.81
2,Ahmedabad,COMMITTED,bau,37488,1054224,644515,307352,102357,61.14,29.15,9.71
3,Ahmedabad,COMMITTED,global-control,9539,265693,159912,79685,26096,60.19,29.99,9.82
4,Ahmedabad,DETOX,bau,1069,7920,6480,1089,351,81.82,13.75,4.43
...,...,...,...,...,...,...,...,...,...,...,...
75,Jaipur,SOFT_CHURN,global-control,3729,21033,12504,8527,2,59.45,40.54,0.01
76,Jaipur,SUSTENANCE,bau,37165,227260,136502,89109,1649,60.06,39.21,0.73
77,Jaipur,SUSTENANCE,global-control,9199,56324,33938,21950,436,60.25,38.97,0.77
78,Jaipur,UNKNOWN,bau,1112,1305,823,482,0,63.07,36.93,0.00


In [42]:
df_final_sample.groupby('city_name')['group_tc'].value_counts(True).unstack(fill_value=0)

group_tc,bau,global-control
city_name,,
Ahmedabad,0.800457,0.199543
Chennai,0.799869,0.200131
Delhi,0.799937,0.200063
Jaipur,0.800339,0.199661


In [43]:
df_final_sample.groupby(['city_name', 'ltr_segment'])['group_tc'].value_counts(True).unstack(fill_value=0)

group_tc                    bau  global-control
city_name ltr_segment                          
Ahmedabad HH           0.800437        0.199563
          PHH          0.800509        0.199491
          UNKNOWN      0.791718        0.208282
Chennai   HH           0.799775        0.200225
          PHH          0.799893        0.200107
          UNKNOWN      0.807489        0.192511
Delhi     HH           0.800023        0.199977
          PHH          0.799891        0.200109
          UNKNOWN      0.799683        0.200317
Jaipur    HH           0.800159        0.199841
          PHH          0.800404        0.199596
          UNKNOWN      0.811087        0.188913

In [44]:
df_final_sample.groupby(['city_name', 'retention_segment'])['group_tc'].value_counts(True).unstack(fill_value=0)

group_tc                          bau  global-control
city_name retention_segment                          
Ahmedabad DORMANT            0.801255        0.198745
          ELITE              0.797502        0.202498
          GOLD               0.801198        0.198802
          HH                 0.800113        0.199887
          INACTIVE           0.796951        0.203049
          PLATINUM           0.797230        0.202770
          PRIME              0.802876        0.197124
          SILVER             0.799935        0.200065
          UNKNOWN            0.791718        0.208282
Chennai   DORMANT            0.799856        0.200144
          ELITE              0.798577        0.201423
          GOLD               0.799476        0.200524
          HH                 0.800048        0.199952
          INACTIVE           0.796997        0.203003
          PLATINUM           0.800876        0.199124
          PRIME              0.800264        0.199736
          SILVER             0.800086        0.199914
          UNKNOWN            0.807489        0.192511
Delhi     DORMANT            0.799669        0.200331
          ELITE              0.800114        0.199886
          GOLD               0.799974        0.200026
          HH                 0.800512        0.199488
          INACTIVE           0.800011        0.199989
          PLATINUM           0.800382        0.199618
          PRIME              0.802622        0.197378
          SILVER             0.799938        0.200062
          UNKNOWN            0.799683        0.200317
Jaipur    DORMANT            0.799881        0.200119
          ELITE              0.800376        0.199624
          GOLD               0.799707        0.200293
          HH                 0.799675        0.200325
          INACTIVE           0.803316        0.196684
          PLATINUM           0.799435        0.200565
          PRIME              0.801611        0.198389
          SILVER             0.802989        0.197011
          UNKNOWN            0.811087        0.188913

### Check

In [45]:
df_final_sample.query("group_tc == 'bau'")["ltr_segment"].value_counts(True)

ltr_segment
PHH        0.675475
HH         0.323117
UNKNOWN    0.001408
Name: proportion, dtype: float64

In [46]:
df_final_sample.query("group_tc == 'global-control'")["ltr_segment"].value_counts(True)

ltr_segment
PHH        0.675513
HH         0.323091
UNKNOWN    0.001395
Name: proportion, dtype: float64

In [47]:
df_final_sample.query("group_tc == 'bau'")["retention_segment"].value_counts(True)

retention_segment
DORMANT     0.487521
GOLD        0.129125
SILVER      0.114297
HH          0.110510
PLATINUM    0.093247
ELITE       0.050887
INACTIVE    0.010091
PRIME       0.002915
UNKNOWN     0.001408
Name: proportion, dtype: float64

In [54]:
df_final_sample.query("group_tc == 'global-control'")["retention_segment"].value_counts(True)

retention_segment
DORMANT     0.487863
GOLD        0.129248
SILVER      0.114007
HH          0.110333
PLATINUM    0.093056
ELITE       0.051088
INACTIVE    0.010130
PRIME       0.002879
UNKNOWN     0.001395
Name: proportion, dtype: float64

In [55]:
df_final_sample.query("group_tc == 'global-control' & city_name == 'Delhi'")["retention_segment"].value_counts(True)

retention_segment
DORMANT     0.494043
HH          0.121569
GOLD        0.121047
SILVER      0.108704
PLATINUM    0.086495
ELITE       0.052840
INACTIVE    0.010262
PRIME       0.003477
UNKNOWN     0.001562
Name: proportion, dtype: float64

In [65]:
df_final_sample.query("group_tc == 'bau' & city_name == 'Delhi'")["retention_segment"].value_counts(True)

retention_segment
DORMANT     0.493217
HH          0.122007
GOLD        0.121074
SILVER      0.108705
PLATINUM    0.086736
ELITE       0.052898
INACTIVE    0.010267
PRIME       0.003536
UNKNOWN     0.001559
Name: proportion, dtype: float64

In [67]:
df_final_sample[(df_final_sample['group_tc'] == 'bau') 
                    &
                (df_final_sample['city_name'] == 'Delhi')]["retention_segment"].value_counts(True)

retention_segment
DORMANT     0.493217
HH          0.122007
GOLD        0.121074
SILVER      0.108705
PLATINUM    0.086736
ELITE       0.052898
INACTIVE    0.010267
PRIME       0.003536
UNKNOWN     0.001559
Name: proportion, dtype: float64

In [74]:
# Get unique values for group_tc and city_name
unique_city_name = df_final_sample['city_name'].unique()
unique_group_tc = df_final_sample['group_tc'].unique()

In [75]:
# Generate all possible combinations
combinations = itertools.product(unique_city_name, unique_group_tc)

In [76]:
# Iterate through combinations and print the results
for city_name_value, group_tc_value in combinations:
    
    print(f"city_name: {city_name_value}, group_tc: {group_tc_value}")
    result = df_final_sample[(df_final_sample['group_tc'] == group_tc_value) & (df_final_sample['city_name'] == city_name_value)]["retention_segment"].value_counts(True)
    print(result)

city_name: Delhi, group_tc: bau
retention_segment
DORMANT     0.493217
HH          0.122007
GOLD        0.121074
SILVER      0.108705
PLATINUM    0.086736
ELITE       0.052898
INACTIVE    0.010267
PRIME       0.003536
UNKNOWN     0.001559
Name: proportion, dtype: float64
city_name: Delhi, group_tc: global-control
retention_segment
DORMANT     0.494043
HH          0.121569
GOLD        0.121047
SILVER      0.108704
PLATINUM    0.086495
ELITE       0.052840
INACTIVE    0.010262
PRIME       0.003477
UNKNOWN     0.001562
Name: proportion, dtype: float64
city_name: Chennai, group_tc: bau
retention_segment
DORMANT     0.460611
GOLD        0.148486
PLATINUM    0.118216
SILVER      0.114521
HH          0.086351
ELITE       0.060449
INACTIVE    0.008182
PRIME       0.002363
UNKNOWN     0.000821
Name: proportion, dtype: float64
city_name: Chennai, group_tc: global-control
retention_segment
DORMANT     0.460648
GOLD        0.148851
PLATINUM    0.117474
SILVER      0.114366
HH          0.086255
ELI

### Export

In [77]:
df_final_sample.head(5)

,customer_id,city_name,customer_mobile,net_orders,bike_service,auto_service,cab_service,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender,group_tc
4326368,6223172182ac75233c329aff,Delhi,9729107902,1,1,0,0,PHH,GOLD,HOOK,UNKNOWN,MALE,bau
1684258,6640bb47f05f5f69ea2cc7e8,Chennai,9884571371,2,0,2,0,HH,DORMANT,DORMANT,UNKNOWN,MALE,bau
7634220,5d21ea613b752c45cf9bda89,Chennai,8838376519,1,1,0,0,PHH,DORMANT,DORMANT,UNKNOWN,MALE,bau
5204993,5dae1ab3341b00418cf314ba,Delhi,8447667147,2,2,0,0,PHH,SILVER,HOOK,UNKNOWN,MALE,bau
574232,66212d931c347208039fd4fd,Chennai,9994441211,4,2,1,1,HH,DORMANT,DORMANT,UNKNOWN,MALE,bau


In [ ]:
# df_train_control['customer_mobile'] = df_train_control['customer_mobile'].astype(float).astype(int)

In [142]:
df_final_sample[df_final_sample['group_tc'] == 'global-control'].to_csv(local_datasource + '/processed/global-control/export.csv', index=False)

In [78]:
df_export = df_final_sample[df_final_sample['group_tc'] == 'global-control']

In [79]:
df_export.shape

(1528494, 13)

In [80]:
df_export.head(5)

,customer_id,city_name,customer_mobile,net_orders,bike_service,auto_service,cab_service,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender,group_tc
7497689,6454c264309f65666beb650b,Delhi,8595191382,1,1,0,0,PHH,SILVER,HOOK,UNKNOWN,UNKNOWN,global-control
3653559,6686dbde25346979e1800542,Chennai,9894117183,1,0,1,0,HH,HH,HANDHOLDING,UNKNOWN,MALE,global-control
3721628,623980b26b6b64a1ac737f9f,Ahmedabad,8104300530,1,1,0,0,PHH,DORMANT,DORMANT,UNKNOWN,MALE,global-control
2556893,613c7d987f5ff756b70258fa,Delhi,7357300227,27,11,16,0,PHH,DORMANT,DORMANT,LINK_AUTO,MALE,global-control
2409427,63801ef42519f5a2f214a4bd,Delhi,9315385605,1,0,1,0,PHH,DORMANT,DORMANT,UNKNOWN,MALE,global-control


In [81]:
df_export\
.groupby(['city_name'])\
.agg(**agg_dict)\
.reset_index()

,city_name,customers,net_orders,bike_service,auto_service,cab_service
0,Ahmedabad,122783,670507,384637,202529,83341
1,Chennai,512888,3751285,1466674,2042720,241891
2,Delhi,727313,4813154,3067458,1093417,652279
3,Jaipur,165510,748142,460530,284136,3476


In [49]:
df_final_sample[['city_name', 'customer_mobile']]

,customer_id,city_name,customer_mobile,net_orders,bike_service,auto_service,cab_service,ltr_segment,retention_segment,lifetime_stage,service_affinity,gender,group_tc
4326368,6223172182ac75233c329aff,Delhi,9729107902,1,1,0,0,PHH,GOLD,HOOK,UNKNOWN,MALE,bau
1684258,6640bb47f05f5f69ea2cc7e8,Chennai,9884571371,2,0,2,0,HH,DORMANT,DORMANT,UNKNOWN,MALE,bau
7634220,5d21ea613b752c45cf9bda89,Chennai,8838376519,1,1,0,0,PHH,DORMANT,DORMANT,UNKNOWN,MALE,bau
5204993,5dae1ab3341b00418cf314ba,Delhi,8447667147,2,2,0,0,PHH,SILVER,HOOK,UNKNOWN,MALE,bau
574232,66212d931c347208039fd4fd,Chennai,9994441211,4,2,1,1,HH,DORMANT,DORMANT,UNKNOWN,MALE,bau
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3143675,6579e1919ca9566fe57f7dd6,Delhi,9250772495,2,2,0,0,HH,HH,HANDHOLDING,UNKNOWN,MALE,global-control
5678032,65f111740f9aeb550a5078c6,Delhi,9310235717,19,17,1,1,PHH,PLATINUM,SOFT_CHURN,ONLY_LINK,MALE,global-control
4308324,6631ef1e8dcc25a0392893ac,Delhi,8920732163,2,0,0,2,HH,DORMANT,DORMANT,UNKNOWN,MALE,global-control
1166382,630cec50744a5c3d14817530,Chennai,6385554331,19,0,13,6,PHH,PLATINUM,COMMITTED,AUTO_CAB,FEMALE,global-control


In [82]:
local_datasource

'/Users/E2074/local-datasets/customer/ads-monetisation/holdout-group'

In [96]:
print(df_export[df_export['city_name'] == 'Jaipur'].customer_id.nunique())
print(df_export[df_export['city_name'] == 'Jaipur'].customer_id.shape[0])

165510
165510


In [97]:
df_export[df_export['city_name'] == 'Jaipur'][['customer_mobile']]\
.to_csv(local_datasource + '/processed/jas_jaipur_holdout_group.csv', header=False, index=False)

In [101]:
df_chennai = df_export[df_export['city_name'] == 'Chennai'][['customer_mobile']]
df_delhi = df_export[df_export['city_name'] == 'Delhi'][['customer_mobile']]

In [118]:
df_chennai.reset_index(drop=True, inplace=True)
df_delhi.reset_index(drop=True, inplace=True)

In [120]:
df_chennai.shape,df_delhi.shape

((512888, 1), (727313, 1))

In [121]:
local_datasource

'/Users/E2074/local-datasets/customer/ads-monetisation/holdout-group'

In [130]:
df_chennai

,customer_mobile
0,9894117183
1,9962437599
2,9790935181
3,9840963627
4,7397259555
...,...
512883,9344221351
512884,9566838593
512885,7538810742
512886,8098372820


In [175]:
def df_split(df,city):
    
    chunk_size = len(df) // 5
    
    for i in range(5):
        start = i * chunk_size
        end = (i + 1) * chunk_size if i < 4 else len(df)
        chunk = df.iloc[start:end]
        chunk.to_csv(f'/Users/E2074/local-datasets/customer/ads-monetisation/holdout-group/processed/jas_{city}_holdout_group_{i+1}.csv', 
                     header=False,
                     index=False)    

In [136]:
df_split(df_chennai,'chennai')

In [176]:
df_split(df_delhi,'delhi')

In [8]:
test_df1 = pd.read_csv(local_datasource+'/processed/jas_delhi_holdout_group_1.csv')
test_df2 = pd.read_csv(local_datasource+'/processed/jas_delhi_holdout_group_2.csv')
test_df3 = pd.read_csv(local_datasource+'/processed/jas_delhi_holdout_group_3.csv')
test_df4 = pd.read_csv(local_datasource+'/processed/jas_delhi_holdout_group_4.csv')
test_df5 = pd.read_csv(local_datasource+'/processed/jas_delhi_holdout_group_5.csv')

In [9]:
test_df1.shape,test_df2.shape,test_df3.shape,test_df4.shape,test_df5.shape

((145461, 1), (145461, 1), (145461, 1), (145461, 1), (145464, 1))

In [179]:
test_df = pd.concat([test_df1,test_df2,test_df3,test_df4, test_df5], axis=0)

In [180]:
test_df.shape

(727308, 5)

In [181]:
test_df

,8595191382,9953968998,9027049041,7982811816,9990911331
0,7.357300e+09,NaN,NaN,NaN,NaN
1,9.315386e+09,NaN,NaN,NaN,NaN
2,9.387284e+09,NaN,NaN,NaN,NaN
3,9.015363e+09,NaN,NaN,NaN,NaN
4,8.595711e+09,NaN,NaN,NaN,NaN
...,...,...,...,...,...
145459,NaN,NaN,NaN,NaN,7.014382e+09
145460,NaN,NaN,NaN,NaN,8.826965e+09
145461,NaN,NaN,NaN,NaN,9.250772e+09
145462,NaN,NaN,NaN,NaN,9.310236e+09
